In [1]:
"""Playground for testing the algebraic tensor module.

Run from the sympy root directory:
    PYTHONPATH=. python sympy/tensor/algebraic/tests/playground_functions.py
"""

import sys
sys.path.insert(0, "/Users/grdi/sympy-tensor-dev/sympy")
from sympy import Symbol, Rational, sqrt, pi, I, symbols
from sympy.core.singleton import S
from sympy.matrices.expressions import MatrixSymbol, MatAdd, MatMul
from sympy.matrices import Matrix, zeros, eye, diag

from sympy.tensor.algebraic import (
    AlgebraicPureTensor,
    AlgebraicTensor,
    AlgebraicZeroTensor,
    algebraic_tensor_product,
    compose_algebraic_pure_tensors,
    compose_algebraic_tensors,
    tensorsimplify,
    ShapeMismatchError,
)

from sympy.printing.latex import latex
from sympy.printing.repr import srepr

# ---------------------------------------------------------------------------
# Commutative symbols
# ---------------------------------------------------------------------------
a, b, c, x, y, z = Symbol("a"), Symbol("b"), Symbol("c"), Symbol("x"), Symbol("y"), Symbol("z")

# ---------------------------------------------------------------------------
# Matrix symbols – various shapes for testing
# ---------------------------------------------------------------------------
# 2x3 matrices
A = MatrixSymbol("A", 2, 3)
A2 = MatrixSymbol("A2", 2, 3)

# 3x4 matrices
B = MatrixSymbol("B", 3, 4)
B2 = MatrixSymbol("B2", 3, 4)

# 3x3 matrices (square, for composition with self-shape)
M = MatrixSymbol("M", 3, 3)
N = MatrixSymbol("N", 3, 3)

# 4x5 matrices
C = MatrixSymbol("C", 4, 5)
C2 = MatrixSymbol("C2", 4, 5)

# 2x2 identity-like square
P = MatrixSymbol("P", 2, 2)
Q = MatrixSymbol("Q", 2, 2)

# 4x2 matrices (for composition chain: 2x3 * 3x4 * 4x2)
R = MatrixSymbol("R", 4, 2)
R2 = MatrixSymbol("R2", 4, 2)

# 2x2 for chaining after R
S_mat = MatrixSymbol("S", 2, 2)
S2 = MatrixSymbol("S2", 2, 2)
def direct_sum0(matrix1, matrix2):
    result = Matrix(zeros(sqrt(len(matrix1))+sqrt(len(matrix2)),sqrt(len(matrix1))+sqrt(len(matrix2))))
    for i in range (sqrt(len(matrix1))):
        for j in range (sqrt(len(matrix1))):
            result[i,j] = matrix1[i,j]
            
    for i in range (sqrt(len(matrix2))):
        for j in range (sqrt(len(matrix2))):
            result[i + sqrt(len(matrix1)),j + sqrt(len(matrix1))] = matrix2[i,j]
    
    return result

### Direct sum of arbitraty number of matrices

def direct_sum(matrices):
    result = matrices[0]
    for i in range (1,len(matrices)):
        result = direct_sum0(result, matrices[i])
    return result


upsilon_u, upsilon_d, upsilon_e, upsilon_nu, upsilon_R = symbols(\
        r"""\Upsilon_u, \Upsilon_d, \Upsilon_e, \Upsilon_\nu, \Upsilon_R""", commutative = False)
upsilont_u, upsilont_d, upsilont_e, upsilont_nu, upsilont_R = symbols(\
        r"""\Upsilon^t_u, \Upsilon^t_d, \Upsilon^t_e, \Upsilon^t_\nu, \Upsilon^t_R""", commutative = False)
upsilonc_u, upsilonc_d, upsilonc_e, upsilonc_nu, upsilonc_R = symbols(\
        r"""\Upsilon^*_u, \Upsilon^*_d, \Upsilon^*_e, \Upsilon^*_\nu, \Upsilon^*_R""", commutative = False)
cupsilon_u, cupsilon_d, cupsilon_e, cupsilon_nu, cupsilon_R = symbols(\
        r"""\overline{\Upsilon}_u, \overline{\Upsilon}_d, \overline{\Upsilon}_e,"""\
       +r"""\overline{\Upsilon}_\nu, \overline{\Upsilon}_R""", commutative = False)


D1 = AlgebraicPureTensor(Matrix([[1,0],[0,0]]), direct_sum([Matrix([1]), Matrix(zeros(3,3))]),Matrix(\
     [[0,0,upsilon_R, upsilonc_nu],[0,0,cupsilon_nu,0],[upsilonc_R,upsilont_nu,0,0],[upsilon_nu,0,0,0]]))

D2 = AlgebraicPureTensor(Matrix([[1,0],[0,0]]),direct_sum([Matrix([0]), eye(3)]), Matrix(\
     [[0,0,0,upsilonc_u],[0,0,cupsilon_u,0],[0,upsilont_u,0,0],[upsilon_u,0,0,0]]))

D3 = AlgebraicPureTensor(Matrix([[0,0],[0,1]]), direct_sum([Matrix([1]),Matrix(zeros(3,3))]),Matrix(\
     [[0,0,0,upsilonc_e],[0,0,cupsilon_e,0],[0,upsilont_e,0,0],[upsilon_e,0,0,0]]))

D4 = AlgebraicPureTensor(Matrix([[0,0],[0,1]]), direct_sum([Matrix([0]),eye(3)]), Matrix(\
     [[0,0,0,upsilonc_d],[0,0,cupsilon_d,0],[0,upsilont_d,0,0],[upsilon_d,0,0,0]]))

Dirac = D1 + D2 + D3 + D4

### Defining representation pi, algebra elements a0,...,a5 and the lists of symbols, list of d(symbol)s, list of -simbols (they are relevant)

def rep(z, w, alpha, beta, gamma, delta, m):
    pi1 = AlgebraicPureTensor(Matrix([[z,0],[0,w]]), eye(4), direct_sum([Matrix([1]), Matrix(zeros(3,3))]))
    pi2 = AlgebraicPureTensor(Matrix([[alpha, beta],[gamma,delta]]), eye(4),\
           direct_sum([Matrix(zeros(3,3)),Matrix([1])]))
    pi3 = AlgebraicPureTensor(eye(2), direct_sum([Matrix([z]), m]), Matrix(diag(0,1,1,0)))
    
    return pi1 + pi2 + pi3



### Defining two algebra elements, a1 and a2:
z0,z1,z2 = symbols(r"""z,z_1,z_2""", complex = True)
w0,w1,w2 = symbols(r"""w,w_1,w_2""", complex = True)
z3,z4,z5 = symbols(r"""z_3,z_4,z_5""", complex = True)
w3,w4,w5 = symbols(r"""w_3,w_4,w_5""", complex = True)
z6,w6 = symbols(r"""z_6, w_6""", complex = True)
z7,z8, w7, w8 = symbols(r"""z_7, z_8, w_7, w_8""", complex = True)

alpha0,alpha1, alpha2,beta0, beta1, beta2 = symbols(r"""\alpha,\alpha_1, \alpha_2,\beta, \beta_1, \beta_2""", complex = True)
gamma0,gamma1, gamma2,delta0, delta1, delta2 = symbols(r"""\gamma,\gamma_1, \gamma_2,\delta, \delta_1, \delta_2""", complex = True)
alpha3,alpha4, alpha5,beta3, beta4, beta5 = symbols(r"""\alpha_3,\alpha_4, \alpha_5,\beta_3, \beta_4, \beta_5""", complex = True)
gamma3,gamma4, gamma5,delta3, delta4, delta5 = symbols(r"""\gamma_3,\gamma_4, \gamma_5,\delta_3, \delta_4, \delta_5""", complex = True)
alpha5, alpha6, beta5, beta6 = symbols(r"""\alpha_5, \alpha_6, \beta_5, \beta_6""", complex = True)
gamma5, gamma6, delta5, delta6 = symbols(r"""\gamma_5, \gamma_6, \delta_5, \delta_6""", complex = True)
alpha7, alpha8, beta7, beta8 = symbols(r"""\alpha_7, \alpha_8, \beta_7, \beta_8""", complex = True)
gamma7, gamma8, delta7, delta8 = symbols(r"""\gamma_7, \gamma_8, \delta_7, \delta_8""", complex = True)


m111,m112,m113, m121,m122,m123, m131,m132,m133 = symbols(\
    r"""m^1_{11},m^1_{12},m^1_{13},m^1_{21},m^1_{22},m^1_{23},m^1_{31},m^1_{32},m^1_{33}""", complex=True)
m011,m012,m013, m021,m022,m023, m031,m032,m033 = symbols(\
    r"""m_{11},m_{12},m_{13},m_{21},m_{22},m_{23},m_{31},m_{32},m_{33}""", complex=True)
m211,m212,m213, m221,m222,m223, m231,m232,m233 = symbols(\
    r"""m^2_{11},m^2_{12},m^2_{13},m^2_{21},m^2_{22},m^2_{23},m^2_{31},m^2_{32},m^2_{33}""", complex=True)
m311,m312,m313, m321,m322,m323, m331,m332,m333 = symbols(\
    r"""m^3_{11},m^3_{12},m^3_{13},m^3_{21},m^3_{22},m^3_{23},m^3_{31},m^3_{32},m^3_{33}""", complex=True)
m411,m412,m413, m421,m422,m423, m431,m432,m433 = symbols(\
    r"""m^4_{11},m^4_{12},m^4_{13},m^4_{21},m^4_{22},m^4_{23},m^4_{31},m^4_{32},m^4_{33}""", complex=True)
m511,m512,m513, m521,m522,m523, m531,m532,m533 = symbols(\
    r"""m^5_{11},m^5_{12},m^5_{13},m^5_{21},m^5_{22},m^5_{23},m^5_{31},m^5_{32},m^5_{33}""", complex=True)
m611,m612,m613, m621,m622,m623, m631,m632,m633 = symbols(\
    r"""m^6_{11},m^6_{12},m^6_{13},m^6_{21},m^6_{22},m^6_{23},m^6_{31},m^6_{32},m^6_{33}""", complex=True)
m711,m712,m713, m721,m722,m723, m731,m732,m733 = symbols(\
    r"""m^7_{11},m^7_{12},m^7_{13},m^7_{21},m^7_{22},m^7_{23},m^7_{31},m^7_{32},m^7_{33}""", complex=True)
m811,m812,m813, m821,m822,m823, m831,m832,m833 = symbols(\
    r"""m^8_{11},m^8_{12},m^8_{13},m^8_{21},m^8_{22},m^8_{23},m^8_{31},m^8_{32},m^8_{33}""", complex=True)

m0 = Matrix([[m011,m012,m013],[m021,m022,m023],[m031,m032,m033]])
m1 = Matrix([[m111,m112,m113],[m121,m122,m123],[m131,m132,m133]])
m2 = Matrix([[m211,m212,m213],[m221,m222,m223],[m231,m232,m233]])
m3 = Matrix([[m311,m312,m313],[m321,m322,m323],[m331,m332,m333]])
m4 = Matrix([[m411,m412,m413],[m421,m422,m423],[m431,m432,m433]])
m5 = Matrix([[m511,m512,m513],[m521,m522,m523],[m531,m532,m533]])
m6 = Matrix([[m611,m612,m613],[m621,m622,m623],[m631,m632,m633]])
m7 = Matrix([[m711,m712,m713],[m721,m722,m723],[m731,m732,m733]])
m8 = Matrix([[m811,m812,m813],[m821,m822,m823],[m831,m832,m833]])


a0 = rep(z0, w0, alpha0, beta0, gamma0, delta0, m0)
a1 = rep(z1, w1, alpha1, beta1, gamma1, delta1, m1)
a2 = rep(z2, w2, alpha2, beta2, gamma2, delta2, m2)
a3 = rep(z3, w3, alpha3, beta3, gamma3, delta3, m3)
a4 = rep(z4, w4, alpha4, beta4, gamma4, delta4, m4)
a5 = rep(z5, w5, alpha5, beta5, gamma5, delta5, m5)
a6 = rep(z6, w6, alpha6, beta6, gamma6, delta6, m6)
a7 = rep(z7, w7, alpha7, beta7, gamma7, delta7, m7)
a8 = rep(z8, w8, alpha8, beta8, gamma8, delta8, m8)

C1,C2,C3,C4,C5,C6,C7,C8,\
C9,C10,C11,C12,C13,C14,C15 = symbols(r"""C_1,C_2,C_3,C_4,C_5,C_6,C_7,C_8,C_9,C_{10},C_{11},C_{12},C_{13},C_{14},C_{15}""", complex = True)
C_m = Matrix([[C7,C8,C9],[C10,C11,C12],[C13,C14,C15]])

general_a = rep(C1,C2,C3,C4,C5,C6,C_m)

test_a1 = rep(0, 0, 1, 0, 0, 0, 0*m1)
test_b1 = rep(0, 0, 0, 1, 0, 0, 0*m1)

test_a2 = rep(0, 0, 0, 1, 0, 0, 0*m1)
test_b2 = rep(0, 1, 0, 0, 0, 0, 0*m1)

minus_simbolovi = [-1*z1,-1*w1,-1*z2,-1*w2,-1*alpha1,-1*alpha2,-1*beta1,-1*beta2,\
             -1*gamma1,-1*gamma2,\
             -1*delta1,-1*delta2,
             -1*m111,-1*m112,-1*m113,-1*m121,-1*m122,-1*m123,-1*m131,-1*m132,-1*m133,\
             -1*m211,-1*m212,-1*m213,-1*m221,-1*m222,-1*m223,-1*m231,-1*m232,-1*m233]


simbolovi0 = [z0, alpha0, beta0, w0, gamma0, delta0,\
              m011,m012,m013, m021,m022,m023, m031,m032,m033]
simbolovi1 = [z1,w1,alpha1,beta1,gamma1,delta1,\
              m111,m112,m113,m121,m122,m123,m131,m132,m133]
simbolovi2 = [z2,w2,alpha2,beta2,gamma2,delta2,\
              m211,m212,m213,m221,m222,m223,m231,m232,m233]
simbolovi3 = [z3,w3,alpha3,beta3,gamma3,delta3,\
              m311,m312,m313,m321,m322,m323,m331,m332,m333]
simbolovi4 = [z4,w4,alpha4,beta4,gamma4,delta4,\
              m411,m412,m413,m421,m422,m423,m431,m432,m433]
simbolovi5 = [z5,w5,alpha5,beta5,gamma5,delta5,\
              m511,m512,m513,m521,m522,m523,m531,m532,m533]
simbolovi6 = [z6,w6,alpha6,beta6,gamma6,delta6,\
              m611,m612,m613,m621,m622,m623,m631,m632,m633]
simbolovi7 = [z7,w7,alpha7,beta7,gamma7,delta7,\
              m711,m712,m713,m721,m722,m723,m731,m732,m733]
simbolovi8 = [z8,w8,alpha8,beta8,gamma8,delta8,\
              m811,m812,m813,m821,m822,m823,m831,m832,m833]

simboloviC = [C1,C2,C3,C4,C5,C6,C7,C8, C9,C10,C11,C12,C13,C14,C15]

simbolovi = simbolovi0 + simbolovi1 + simbolovi2 + simbolovi3 + simbolovi4 + simbolovi5 + simbolovi6 + simbolovi7 + simbolovi8
simbolovi08 = simbolovi0 + simbolovi1 + simbolovi2 + simbolovi3 + simbolovi4 + simbolovi5 + simbolovi6 + simbolovi7 + simbolovi8

da1 = Dirac*a1 - a1*Dirac
#da1 = da1.expand()
da1 = da1.simplify()

print(len(da1.args))

da1 = da1
da1.display()

8


<IPython.core.display.Latex object>

In [2]:
P = MatrixSymbol("P", 2, 2)
Q = MatrixSymbol("Q", 2, 2)
testoni_100 = AlgebraicPureTensor(2+x,P+Q,Q)
print(testoni_100)
testoni_100.expand().display()

(x + 2)*(P + Q) ⊗ Q


<IPython.core.display.Latex object>

In [3]:
print(testoni_100._repr_latex_())

$\displaystyle \left(x + 2\right) \left(P + Q\right) \otimes Q$


In [4]:
t1 = AlgebraicPureTensor(x, P, Q)
t2 = AlgebraicPureTensor(2, P, Q)


testoni_300 = t1*t2

test1 = AlgebraicTensor(t1, t2)
test2 = t1 + t2

In [5]:
print(test1)

(x + 2)*P ⊗ Q


In [6]:
print(test2)

(x + 2)*P ⊗ Q


In [7]:
srepr(algebraic_tensor_product(2, P))

"MatMul(Integer(2), MatrixSymbol(Str('P'), Integer(2), Integer(2)))"

In [8]:
srepr(2*P)

"MatMul(Integer(2), MatrixSymbol(Str('P'), Integer(2), Integer(2)))"

In [9]:
a1.display()

<IPython.core.display.Latex object>

In [10]:
da1 = Dirac*a1 - a1*Dirac
dada = algebraic_tensor_product(da1.expand(), da1.expand()).expand()

dada_subset = dada.args[0]
for i in range (10):
    dada_subset += dada.args[i+1]

In [11]:
len(dada_subset.simplify().args)

24

In [12]:
dada.args[0]

Matrix([
[       0,        0],
[\gamma_1, \delta_1]]) ⊗ Matrix([
[1, 0, 0, 0],
[0, 0, 0, 0],
[0, 0, 0, 0],
[0, 0, 0, 0]]) ⊗ Matrix([
[0, 0, 0, \Upsilon^*_e],
[0, 0, 0,            0],
[0, 0, 0,            0],
[0, 0, 0,            0]]) ⊗ Matrix([
[       0,        0],
[\gamma_1, \delta_1]]) ⊗ Matrix([
[1, 0, 0, 0],
[0, 0, 0, 0],
[0, 0, 0, 0],
[0, 0, 0, 0]]) ⊗ Matrix([
[0, 0, 0, \Upsilon^*_e],
[0, 0, 0,            0],
[0, 0, 0,            0],
[0, 0, 0,            0]])

In [13]:
len(dada.args)

324

In [14]:
simped = dada.simplify()

In [15]:
len(simped.simplify().args)

64

In [16]:
simped.display()

<IPython.core.display.Latex object>

In [17]:
simped.args[-1]

(\delta_1 - w_1)**2*Matrix([
[0, 0],
[0, 1]]) ⊗ Matrix([
[0, 0, 0, 0],
[0, 1, 0, 0],
[0, 0, 1, 0],
[0, 0, 0, 1]]) ⊗ Matrix([
[          0, 0, 0, \Upsilon^*_d],
[          0, 0, 0,            0],
[          0, 0, 0,            0],
[-\Upsilon_d, 0, 0,            0]]) ⊗ Matrix([
[0, 0],
[0, 1]]) ⊗ Matrix([
[0, 0, 0, 0],
[0, 1, 0, 0],
[0, 0, 1, 0],
[0, 0, 0, 1]]) ⊗ Matrix([
[          0, 0, 0, \Upsilon^*_d],
[          0, 0, 0,            0],
[          0, 0, 0,            0],
[-\Upsilon_d, 0, 0,            0]])

In [18]:
da1 = Dirac*a1 - a1*Dirac
da1.expand().display()

<IPython.core.display.Latex object>

In [45]:
(da1 - da1.expand()).simplify().T

0_{(2x2), (4x4), (4x4)}

In [20]:
simped.subs({z1 : w1})

(\delta_1 - w_1)**2*Matrix([
[0, 0],
[0, 1]]) ⊗ Matrix([
[0, 0, 0, 0],
[0, 1, 0, 0],
[0, 0, 1, 0],
[0, 0, 0, 1]]) ⊗ Matrix([
[          0, 0, 0, \Upsilon^*_d],
[          0, 0, 0,            0],
[          0, 0, 0,            0],
[-\Upsilon_d, 0, 0,            0]]) ⊗ Matrix([
[0, 0],
[0, 1]]) ⊗ Matrix([
[0, 0, 0, 0],
[0, 1, 0, 0],
[0, 0, 1, 0],
[0, 0, 0, 1]]) ⊗ Matrix([
[          0, 0, 0, \Upsilon^*_d],
[          0, 0, 0,            0],
[          0, 0, 0,            0],
[-\Upsilon_d, 0, 0,            0]]) + \gamma_1*(\delta_1 - w_1)*Matrix([
[0, 0],
[0, 1]]) ⊗ Matrix([
[0, 0, 0, 0],
[0, 1, 0, 0],
[0, 0, 1, 0],
[0, 0, 0, 1]]) ⊗ Matrix([
[          0, 0, 0, \Upsilon^*_d],
[          0, 0, 0,            0],
[          0, 0, 0,            0],
[-\Upsilon_d, 0, 0,            0]]) ⊗ Matrix([
[0, 0],
[1, 0]]) ⊗ Matrix([
[0, 0, 0, 0],
[0, 1, 0, 0],
[0, 0, 1, 0],
[0, 0, 0, 1]]) ⊗ Matrix([
[          0, 0, 0, \Upsilon^*_d],
[          0, 0, 0,            0],
[          0, 0, 0,            0],


In [21]:
simped.doit()

\gamma_1**2*Matrix([
[0, 0],
[1, 0]]) ⊗ Matrix([
[1, 0, 0, 0],
[0, 0, 0, 0],
[0, 0, 0, 0],
[0, 0, 0, 0]]) ⊗ Matrix([
[            0, 0, 0, \Upsilon^*_e],
[            0, 0, 0,            0],
[            0, 0, 0,            0],
[-\Upsilon_\nu, 0, 0,            0]]) ⊗ Matrix([
[0, 0],
[1, 0]]) ⊗ Matrix([
[1, 0, 0, 0],
[0, 0, 0, 0],
[0, 0, 0, 0],
[0, 0, 0, 0]]) ⊗ Matrix([
[            0, 0, 0, \Upsilon^*_e],
[            0, 0, 0,            0],
[            0, 0, 0,            0],
[-\Upsilon_\nu, 0, 0,            0]]) + \gamma_1*(\delta_1 - w_1)*Matrix([
[0, 0],
[1, 0]]) ⊗ Matrix([
[1, 0, 0, 0],
[0, 0, 0, 0],
[0, 0, 0, 0],
[0, 0, 0, 0]]) ⊗ Matrix([
[            0, 0, 0, \Upsilon^*_e],
[            0, 0, 0,            0],
[            0, 0, 0,            0],
[-\Upsilon_\nu, 0, 0,            0]]) ⊗ Matrix([
[0, 0],
[0, 1]]) ⊗ Matrix([
[1, 0, 0, 0],
[0, 0, 0, 0],
[0, 0, 0, 0],
[0, 0, 0, 0]]) ⊗ Matrix([
[          0, 0, 0, \Upsilon^*_e],
[          0, 0, 0,            0],
[          0, 0, 0,

In [22]:
print(testoni_100)

(x + 2)*(P + Q) ⊗ Q


In [23]:
simped.args[0].diff(gamma1)

2*\gamma_1*Matrix([
[0, 0],
[1, 0]]) ⊗ Matrix([
[1, 0, 0, 0],
[0, 0, 0, 0],
[0, 0, 0, 0],
[0, 0, 0, 0]]) ⊗ Matrix([
[            0, 0, 0, \Upsilon^*_e],
[            0, 0, 0,            0],
[            0, 0, 0,            0],
[-\Upsilon_\nu, 0, 0,            0]]) ⊗ Matrix([
[0, 0],
[1, 0]]) ⊗ Matrix([
[1, 0, 0, 0],
[0, 0, 0, 0],
[0, 0, 0, 0],
[0, 0, 0, 0]]) ⊗ Matrix([
[            0, 0, 0, \Upsilon^*_e],
[            0, 0, 0,            0],
[            0, 0, 0,            0],
[-\Upsilon_\nu, 0, 0,            0]])

In [24]:
testoni_100.diff(x)

(P + Q) ⊗ Q

In [25]:
testoni_100

(x + 2)*(P + Q) ⊗ Q

In [26]:
srepr(testoni_100.diff(x))

'AlgebraicPureTensor((P + Q), Q)'

In [27]:
display(testoni_100.diff(x))

(P + Q) ⊗ Q

In [28]:
simped.args[0].args[3].conjugate()

Matrix([
[                       0, 0, 0, conjugate(\Upsilon^*_e)],
[                       0, 0, 0,                       0],
[                       0, 0, 0,                       0],
[-conjugate(\Upsilon_\nu), 0, 0,                       0]])

In [29]:
simped.args[0].T.conjugate()

conjugate(\gamma_1)**2*Matrix([
[0, 1],
[0, 0]]) ⊗ Matrix([
[1, 0, 0, 0],
[0, 0, 0, 0],
[0, 0, 0, 0],
[0, 0, 0, 0]]) ⊗ Matrix([
[                      0, 0, 0, -conjugate(\Upsilon_\nu)],
[                      0, 0, 0,                        0],
[                      0, 0, 0,                        0],
[conjugate(\Upsilon^*_e), 0, 0,                        0]]) ⊗ Matrix([
[0, 1],
[0, 0]]) ⊗ Matrix([
[1, 0, 0, 0],
[0, 0, 0, 0],
[0, 0, 0, 0],
[0, 0, 0, 0]]) ⊗ Matrix([
[                      0, 0, 0, -conjugate(\Upsilon_\nu)],
[                      0, 0, 0,                        0],
[                      0, 0, 0,                        0],
[conjugate(\Upsilon^*_e), 0, 0,                        0]])

In [30]:
simped.args[0]

\gamma_1**2*Matrix([
[0, 0],
[1, 0]]) ⊗ Matrix([
[1, 0, 0, 0],
[0, 0, 0, 0],
[0, 0, 0, 0],
[0, 0, 0, 0]]) ⊗ Matrix([
[            0, 0, 0, \Upsilon^*_e],
[            0, 0, 0,            0],
[            0, 0, 0,            0],
[-\Upsilon_\nu, 0, 0,            0]]) ⊗ Matrix([
[0, 0],
[1, 0]]) ⊗ Matrix([
[1, 0, 0, 0],
[0, 0, 0, 0],
[0, 0, 0, 0],
[0, 0, 0, 0]]) ⊗ Matrix([
[            0, 0, 0, \Upsilon^*_e],
[            0, 0, 0,            0],
[            0, 0, 0,            0],
[-\Upsilon_\nu, 0, 0,            0]])

In [31]:
A = Matrix([[1,2,3],[4,5,6]])

In [32]:
A.shape

(2, 3)

In [33]:
A

Matrix([
[1, 2, 3],
[4, 5, 6]])

In [34]:
zm = AlgebraicZeroTensor((3, 4))  # dispatches to ZeroMatrix
azt = AlgebraicZeroTensor(((3, 4), (4, 5)))

# ZeroMatrix.__mul__ routes through MatMul (not compose_algebraic_tensors),
# producing a MatMul result rather than raising an error.
result = zm*azt

/Users/grdi/sympy-tensor-dev/sympy/sympy/matrices/expressions/matmul.py:129: SymPyDeprecationWarning: 

Using non-Expr arguments in Mul is deprecated (in this case, one of
the arguments has type 'AlgebraicZeroTensor').

If you really did intend to use a multiplication or addition operation with
this object, use the * or + operator instead.

See https://docs.sympy.org/latest/explanation/active-deprecations.html#non-expr-args-deprecated
for details.

This has been deprecated since SymPy version 1.7. It
will be removed in a future version of SymPy.

  coeff = Mul(*scalars)


In [35]:
print(srepr(result))

ZeroMatrix(Integer(3), Integer(4))


In [36]:
azt.subs({z1:w1}).display()

<IPython.core.display.Latex object>

In [37]:
M = Matrix([[x, x**2], [1, x]])
N = MatrixSymbol("N", 2, 3)
T2 = AlgebraicPureTensor(M, N)
T2.diff(x)  # doctest: +SKIP

Matrix([
[1, 2*x],
[0,   1]]) ⊗ N

In [38]:
M2 = Matrix([[x, 1], [0, x]])
T3 = AlgebraicPureTensor(x, M2, N)
T3_diff = T3.diff(x)
len(T3_diff.args)

2

In [39]:
T3_diff

x*Matrix([
[1, 0],
[0, 1]]) ⊗ N + Matrix([
[x, 1],
[0, x]]) ⊗ N

In [40]:
M2

Matrix([
[x, 1],
[0, x]])

In [41]:
N.diff(x)

0

In [42]:
T3

x*Matrix([
[x, 1],
[0, x]]) ⊗ N